# LoRA / QLoRA Fine-Tuning — Colab Entry Point

This notebook is the **single training entry point** for LoRA and QLoRA runs.
All training logic lives in `src/finetune.py`; this notebook handles Colab-specific
setup (Drive mount, repo access, dependency install) and then delegates to that script.

**Switch between LoRA and QLoRA by changing `CONFIG` in cell 4.**

Runtime required: GPU (T4 or better). CPU runtime will fail for QLoRA.

## 1. Mount Google Drive and set repo path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path to where the repo lives on your Drive
REPO_PATH = '/content/drive/MyDrive/FinalProjectMSCS'

import os
os.chdir(REPO_PATH)
print('Working directory:', os.getcwd())

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Confirm data files are present

In [ ]:
from pathlib import Path

required = [
    'data/processed/qa_sft_train.jsonl',
    'data/processed/qa_sft_val.jsonl',
    'data/processed/benchmark_questions_curated.jsonl',
    'configs/lora.yaml',
    'configs/qlora.yaml',
]
for p in required:
    status = 'OK' if Path(p).exists() else 'MISSING'
    print(f'{status}  {p}')

## 4. Choose config: LoRA or QLoRA

Set `CONFIG` to either `configs/lora.yaml` or `configs/qlora.yaml`.

> **Current experiment: QLoRA** — the cell below is pre-set to `configs/qlora.yaml`.
> Outputs will be written to `outputs/ft/qlora_tinyllama`.
> To re-run LoRA, change `CONFIG` back to `configs/lora.yaml`.

In [ ]:
CONFIG = 'configs/qlora.yaml'  # change to 'configs/lora.yaml' to re-run LoRA
print('Using config:', CONFIG)

## 5. Dry-run validation (no training, no GPU cost)

In [ ]:
!python src/finetune.py --config {CONFIG} --dry-run

## 6. Training run

Only execute this cell after the dry-run above passes cleanly.

In [ ]:
!python src/finetune.py --config {CONFIG}

## 7. Mirror adapter to local Colab runtime

**Storage summary:**

| Location | Persistent? | Purpose |
|---|---|---|
| `outputs/ft/<run>/` on Drive | Yes — survives session end | Source of truth for all adapter weights |
| `/content/outputs/ft/<run>/` | No — wiped when session ends | Local inspection during the session |

The local path mirrors the Drive path structure exactly (e.g. `outputs/ft/qlora_tinyllama` → `/content/outputs/ft/qlora_tinyllama`), so both locations look identical in the file browser. Re-run this cell any time you want to refresh the local copy.

In [ ]:
import shutil, yaml
from pathlib import Path

ADAPTER_DIR = yaml.safe_load(Path(CONFIG).read_text())['training']['output_dir']
src = Path(ADAPTER_DIR)

# Mirror to /content/outputs/ft/<run>/ — same relative structure as Drive
dst = Path('/content') / ADAPTER_DIR
dst.parent.mkdir(parents=True, exist_ok=True)

if not (src / 'adapter_config.json').exists():
    print(f"[SKIP] No adapter at '{src}'. Run the training cell first.")
else:
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

    print(f"Local copy : {dst}")
    print(f"Drive source: {src}")
    print()
    for entry in sorted(dst.rglob('*')):
        rel = entry.relative_to(dst)
        indent = '  ' * (len(rel.parts) - 1)
        size = f'  {entry.stat().st_size:>12,} B' if entry.is_file() else '/'
        print(f"  {indent}{rel.name}{size}")

## 8. Quick inference spot-check

Run a single prompt through the adapter to verify it responds before running the full benchmark evaluation.

In [ ]:
import yaml, os
from pathlib import Path

cfg = yaml.safe_load(Path(CONFIG).read_text())
ADAPTER_DIR = cfg['training']['output_dir']
PROMPT = 'What are the admission requirements for the CS graduate program?'

if not (Path(ADAPTER_DIR) / 'adapter_config.json').exists():
    print(f"[SKIP] No adapter found at '{ADAPTER_DIR}'. Run the training cell first.")
else:
    os.system(
        f'python src/ft_infer.py'
        f' --config {CONFIG}'
        f' --adapter-dir {ADAPTER_DIR}'
        f' --prompt "{PROMPT}"'
        f' --max-new-tokens 150'
    )

## 9. Evaluate fine-tuned adapter

Runs `src/ft_eval.py` against the 7-question curated held-out benchmark.
Produces two files under `outputs/ft/`:

| File | Content |
|---|---|
| `ft_eval_<run>_<ts>.jsonl` | Per-question trace with generated answer + metrics |
| `ft_eval_summary_<run>_<ts>.json` | Compact summary (n, avg overlap, short/empty counts) |

`weak_reference_overlap` uses the identical token-overlap formula as the locked RAG baseline,
so scores are directly comparable. No retrieval context is injected.

In [ ]:
import yaml, json, subprocess, sys
from pathlib import Path

ADAPTER_DIR = yaml.safe_load(Path(CONFIG).read_text())['training']['output_dir']
BENCHMARK   = 'data/processed/benchmark_questions_curated.jsonl'
OUTPUT_DIR  = 'outputs/ft'

print(f"Adapter  : {ADAPTER_DIR}")
print(f"Benchmark: {BENCHMARK}")
print(f"Output   : {OUTPUT_DIR}")

if not (Path(ADAPTER_DIR) / 'adapter_config.json').exists():
    print(f"[SKIP] No adapter at '{ADAPTER_DIR}'. Run the training cell first.")
else:
    result = subprocess.run(
        [sys.executable, 'src/ft_eval.py',
         '--config',         CONFIG,
         '--adapter-dir',    ADAPTER_DIR,
         '--benchmark',      BENCHMARK,
         '--output-dir',     OUTPUT_DIR,
         '--max-new-tokens', '200'],
        capture_output=True,
        text=True,
        check=False,
    )

    # Always print captured output so errors are visible in the cell.
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)

    print(f"Exit code: {result.returncode}")

    summaries = sorted(Path(OUTPUT_DIR).glob(f'ft_eval_summary_{Path(ADAPTER_DIR).name}_*.json'))
    if summaries:
        print(f"\nSummary file: {summaries[-1]}")
        print(json.dumps(json.loads(summaries[-1].read_text()), indent=2))
    elif result.returncode == 0:
        print("[WARN] Script exited 0 but no summary file found — check output_dir path.")
    else:
        print("[ERROR] Evaluation failed — full output shown above.")

## 10. Save adapter to Drive

The adapter is already written to `outputs/ft/` inside the repo directory on Drive.
This cell creates an explicit timestamped backup copy.

In [ ]:
import shutil
from datetime import datetime, timezone
from pathlib import Path

if not (Path(ADAPTER_DIR) / 'adapter_config.json').exists():
    print(f"[SKIP] No adapter at '{ADAPTER_DIR}'. Run training first.")
else:
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    backup_name = f"{ADAPTER_DIR.rstrip('/').split('/')[-1]}_{timestamp}"
    backup_path = f'/content/drive/MyDrive/FinalProjectMSCS_adapters/{backup_name}'
    shutil.copytree(ADAPTER_DIR, backup_path)
    print('Adapter saved to:', backup_path)